<!--
Copyright 2026 Yaroslav Mariukha
SPDX-License-Identifier: Apache-2.0
-->


# FPGA Verification Library Roadmap

`fpga-verification` provides reusable numeric formats, video and wire-protocol codecs, cocotb/pyuvm verification components, simulation runners, and Intel Quartus hardware-in-the-loop helpers. This notebook is a short tour and a map to the focused tutorials.

## Installation

Install a released package:

```bash
python -m pip install fpga-verification
```

For library development, install the repository in editable mode:

```bash
python -m pip install -e .
```

Python-only codecs can be explored directly in Jupyter. Bus functional models and agents require a cocotb simulation, while HIL examples require Intel Quartus System Console and connected hardware.

## Package map

```text
fpga_verification
├── formats                         integer and fixed-point raw words
├── video                           numpy frames and payload packing
├── protocols.avalon_st.intel_video Intel VIP packet model and codecs
├── sim
│   ├── buses                       Avalon-ST and Avalon-MM BFMs
│   ├── agents                      pyuvm agents and monitors
│   ├── bfms                        Intel DMA black-box model
│   ├── models                      reference predictors
│   ├── scoreboards                 output checking
│   ├── stream_metrics              latency and throughput analysis
│   ├── runners                     RTL and Intel component runners
│   └── platform_designer           generated-system runners
└── hil.intel                       Quartus System Console access
```

## How the layers fit together

```text
numeric samples
      │
      ▼
numpy video frame ── VideoPayloadCodec ── video symbols
      │                                      │
      └────── IntelVIPFrameCodec ────────────┤
                                             ▼
                                    Intel VIP packets
                                             │
                                             ▼
                                  Avalon-ST source/sink
                                             │
                                             ▼
                                            DUT
```

The protocol-neutral layers are useful in unit tests and data preparation. Simulation components add clocks, handshaking, reset handling, monitoring, prediction, and scoreboarding.

## Tutorial map

| Notebook | Main subject | Environment |
|---|---|---|
| `01_numeric_formats.ipynb` | `UIntFormat` and `QFormat` | Python |
| `02_video_frames.ipynb` | Frames, image generation, and payload conversion | Python |
| `03_avalon_st_bus.ipynb` | Avalon-ST source, sink, monitor, and beats | cocotb simulator |
| `04_avalon_mm_bus.ipynb` | Avalon-MM master, slave, memory, and monitor | cocotb simulator |
| `05_intel_vip_packets_and_agent.ipynb` | Intel VIP packets, codecs, and checking | Python, then pyuvm |
| `06_intel_dma_bfm.ipynb` | Intel DMA command/data model | cocotb simulator |
| `07_hil_system_console.ipynb` | Memory access on real Intel FPGA hardware | Quartus + hardware |
| `08_simulation_runners.ipynb` | RTL, Intel component, and Platform Designer runners | HDL tools |
| `09_vip_verification_components.ipynb` | VIP predictor, agent, and scoreboard architecture | pyuvm simulator |
| `10_stream_performance_metrics.ipynb` | Packet latency, gaps, overlap, and efficiency | Python/cocotb |

## Suggested paths

**Video pipeline verification**

```text
01 → 02 → 05 → 03 → 09 → 10
```

**Register-controlled component**

```text
01 → 04 → 08
```

**DMA and memory traffic**

```text
03 → 06 → 10
```

**Hardware-in-the-loop validation**

```text
01 → 02 → 07
```

## Five-minute Python tour

The following example creates a small RGB frame, converts it to Intel VIP packets, and reconstructs the original numpy array. No simulator is required.

In [1]:
from fpga_verification import (
    FrameSize,
    ImageGenerator,
    IntelVIPFrameCodec,
    VideoFormat,
    compare_frames,
)

fmt = VideoFormat(
    bits_per_color=8,
    number_of_color_planes=3,
    pixels_in_parallel=1,
)
size = FrameSize(width=4, height=3)
frame = ImageGenerator(fmt, rng=1).horizontal_ramp(size)

codec = IntelVIPFrameCodec(fmt)
packets = codec.frame_to_packets(frame, size)
decoded = codec.packets_to_frame(packets, size)

compare_frames(decoded, frame)
print([packet.packet_type.name for packet in packets])
print(f"Decoded frame shape: {decoded.shape}")

['CONTROL', 'VIDEO']
Decoded frame shape: (3, 4, 3)
